# M4 exploratory analysis: do vessels slow down or wait when the sea is rough?
Reads the M2 tables and the M3 cleaned layer **read-only**, writes every figure to `figures/`.
Run on a fresh build: `python src/load.py && python src/clean.py && python src/validate.py`, then Restart & Run All.
Every parameter is in the first code cell. Every query states its flag filter.

In [ ]:
import sys, types
from collections import Counter, defaultdict
from itertools import combinations
from datetime import datetime, timedelta
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sql" / "schema.sql").exists())
sys.path.insert(0, str(ROOT / "src"))
import explore as E
from db import table_digest
from clean import clean_digest

PARAMS = dict(
    TZ_NAME="Europe/Istanbul",        # Turkey is UTC+3 all year; port and vessel behaviour follow this clock
    WAIT_KN=0.5,                      # "waiting" = sog below this many knots
    MATCH_RADIUS_DEG=0.5,             # an AIS position belongs to a wave point if its nearest point is this close
    RADIUS_SENSITIVITY=(0.5, 1.0, 1.5),
    COS2=0.587,                       # cos(40 deg)^2: a degree of longitude is shorter than a degree of latitude here
    SOG_FLAGS_MAIN=("ok", "unusual_high"), SOG_FLAGS_STRICT=("ok",),
    ROW_FLAGS_MAIN=("ok", "nav_status_conflicts_with_sog"), ROW_FLAGS_STRICT=("ok",),
    WAVE_FLAGS_MAIN=("ok", "unusual_high", "stuck_value"), WAVE_FLAGS_STRICT=("ok",),
    MIN_HOURS_PER_DAY=18,             # a local-day wave mean needs at least this many of 24 hours
    MIN_POSITIONS_PER_GROUP=10,       # a share over a group needs at least this many positions
    MIN_POSITIONS_FOR_CORR=8,         # below this a coefficient is printed but not interpreted
    HIST_BINS_KN=(0.5, 1.0, 2.0), HIST_BINS_M=(0.02, 0.04, 0.1),
    NORTH_LAT=38.0,                   # positions at or north of this latitude are in the area of our wave points
    HOLIDAYS={"2026-10-29": "Republic Day"},
    N_BOOT=5000, SEED=20261115, CI=95,
)
P = types.SimpleNamespace(**PARAMS)
TZ = E.local_tz(P.TZ_NAME)
con = E.connect_ro()
RAW_BEFORE, CLEAN_BEFORE = table_digest(con), clean_digest(con)
WAVE_F3, SOG_F2, ROW_F2 = E.pad(P.WAVE_FLAGS_MAIN, 3), E.pad(P.SOG_FLAGS_MAIN, 2), E.pad(P.ROW_FLAGS_MAIN, 2)
print("read-only connection; hashed", len(RAW_BEFORE), "M2 tables and", len(CLEAN_BEFORE), "M3 tables")

## 2.1 Raw versus cleaned: the key number
Project question: **when the sea is rougher, do vessels near our wave points slow down and wait more?**
Key number: the share of AIS positions that are *waiting* (sog below 0.5 kn) among positions whose nearest wave point is
within 0.5 degrees, in an hour with a wave reading. Computed on the M2 tables (no flag filter) and on the cleaned layer.

In [ ]:
raw_sql, clean_sql = E.read_query("m4_key_raw.sql"), E.read_query("m4_key_clean.sql")
print("---- RAW query (sql/queries/m4_key_raw.sql) ----"); print(raw_sql)
print("---- CLEANED query (sql/queries/m4_key_clean.sql) ----"); print(clean_sql)
r2 = P.MATCH_RADIUS_DEG ** 2
raw = E.query(con, raw_sql, (P.COS2, P.WAIT_KN, P.WAIT_KN, r2))[0]
clean_params = (P.COS2, *SOG_F2, *ROW_F2, P.WAIT_KN, P.WAIT_KN, r2, *WAVE_F3)
cl = E.query(con, clean_sql, clean_params)[0]
print("RAW     (M2 tables, no flag filter)                :", raw)
print("CLEANED (sog %s, row %s, wave %s):" % (P.SOG_FLAGS_MAIN, P.ROW_FLAGS_MAIN, P.WAVE_FLAGS_MAIN), cl)
print()
print("How much cleaning changed OTHER numbers (heading is circular, so its mean is only an illustration of the 511 code):")
E.show([dict(version="raw, all rows", **E.query(con, "SELECT COUNT(true_heading) AS n, ROUND(AVG(true_heading), 1) AS mean_heading FROM ais_position")[0]),
        dict(version="cleaned, true_heading_flag = 'ok'", **E.query(con, "SELECT COUNT(true_heading) AS n, ROUND(AVG(true_heading), 1) AS mean_heading FROM ais_position_clean WHERE true_heading_flag = 'ok'")[0])])
E.show([dict(version="raw wave_height, all rows", **E.query(con, "SELECT COUNT(wave_height) AS n, ROUND(AVG(wave_height), 4) AS mean_wave_m FROM wave_reading")[0]),
        dict(version="cleaned, flags ok + unusual_high + stuck_value", **E.query(con, "SELECT COUNT(wave_height) AS n, ROUND(AVG(wave_height), 4) AS mean_wave_m FROM wave_reading_clean WHERE wave_height_flag IN ('ok', 'unusual_high', 'stuck_value')")[0])])

## Data used below: every AIS position with its nearest wave point and the wave of its own hour (cleaned layer, main flag filter)

In [ ]:
MP = E.query(con, E.read_query("m4_matched_positions.sql"), (P.COS2, *SOG_F2, *ROW_F2, *WAVE_F3))
for r in MP:
    t = E.parse_utc(r["observed_at"])
    r["window"] = t.strftime("%d %b %H") + "Z"          # collection window = UTC date + hour
    r["utc_date"] = t.strftime("%Y-%m-%d")
    r["local"] = t.astimezone(TZ)
    r["dist_deg"] = r["dist2"] ** 0.5
WINDOWS = sorted({r["window"] for r in MP}, key=lambda w: min(x["observed_at"] for x in MP if x["window"] == w))
print(len(MP), "positions;", len({r["vessel_key"] for r in MP}), "vessels; windows (UTC):", WINDOWS)
print("positions per window:", Counter(r["window"] for r in MP))

## 2.2 The question, and the summary that answers it
The question asks about a **rate** (how many vessels are waiting) and about **typical speed**. We report the share of positions
below 0.5 kn because "waiting" is a state, not an average speed: a mean speed of 5 kn can hide half the vessels standing still.
Alternative summaries of the same variable: median, mean and 90th percentile of `sog`. We check whether they rank the groups the same way.

In [ ]:
def summ(rows):
    s = np.array([r["sog"] for r in rows], float)
    if len(s) == 0:
        return dict(n=0)
    return dict(n=len(s), pct_waiting=round(100 * float((s < P.WAIT_KN).mean()), 1), mean_kn=round(float(s.mean()), 2),
                median_kn=round(float(np.median(s)), 2), p90_kn=round(float(np.percentile(s, 90)), 2))
by_window = {w: summ([r for r in MP if r["window"] == w]) for w in WINDOWS}
print("ALL positions by collection window (flags: sog %s, row %s):" % (P.SOG_FLAGS_MAIN, P.ROW_FLAGS_MAIN))
E.show([dict(window=w, **v) for w, v in by_window.items()])
usable = [w for w in WINDOWS if by_window[w]["n"] >= P.MIN_POSITIONS_PER_GROUP]
print("windows with at least", P.MIN_POSITIONS_PER_GROUP, "positions (coverage rule):", usable)
def ranking(key, reverse):
    return sorted(usable, key=lambda w: by_window[w][key], reverse=reverse)
if len(usable) >= 2:
    r_share, r_med, r_mean, r_p90 = ranking("pct_waiting", True), ranking("median_kn", False), ranking("mean_kn", False), ranking("p90_kn", False)
    print("slowest-first ranking by share waiting:", r_share)
    print("slowest-first ranking by median speed :", r_med, "same as share" if r_med == r_share else "DIFFERENT")
    print("slowest-first ranking by mean speed   :", r_mean, "same as share" if r_mean == r_share else "DIFFERENT")
    print("slowest-first ranking by p90 speed    :", r_p90, "same as share" if r_p90 == r_share else "DIFFERENT")
MATCHED = [r for r in MP if r["dist_deg"] <= P.MATCH_RADIUS_DEG and r["wave_height"] is not None]
by_point = defaultdict(list)
for r in MATCHED: by_point[r["point_key"]].append(r)
print("\nMATCHED positions (nearest wave point within %.1f deg, wave reading present): n = %d" % (P.MATCH_RADIUS_DEG, len(MATCHED)))
E.show([dict(point_key=k, **summ(v)) for k, v in sorted(by_point.items())] or [])

## 2.3 Shape
Speed: three bin widths tried (0.5, 1 and 2 kn), plus an ECDF per window. Waves: bin widths 0.02, 0.04, 0.1 m (0.02 m is the model's own grid),
log scale on the counts because the distribution is heavily right-skewed, plus an ECDF per wave point.

In [ ]:
sog = np.array([r["sog"] for r in MP], float)
n_zero = int((sog < P.WAIT_KN).sum())
fig, axes = plt.subplots(2, 2, figsize=(10, 6.2), sharex=False)
xmax = float(np.ceil(sog.max() / 2) * 2) if len(sog) else 2
for ax, bw in zip(axes.ravel()[:3], P.HIST_BINS_KN):
    ax.hist(sog, bins=np.arange(0, xmax + bw, bw), color="#4477aa", edgecolor="white")
    ax.axvline(P.WAIT_KN, color="#cc3311", lw=1, ls="--")
    ax.set_xlim(0, xmax); ax.set_title("bin width %.1f kn" % bw, fontsize=9, loc="left")
    ax.set_xlabel("speed over ground (knots)"); ax.set_ylabel("positions per bin")
ax = axes.ravel()[3]
for w in WINDOWS:
    x, y = E.ecdf([r["sog"] for r in MP if r["window"] == w]); ax.step(x, y, where="post", label="%s (n=%d)" % (w, len(x)))
ax.axvline(P.WAIT_KN, color="#cc3311", lw=1, ls="--", label="waiting threshold %.1f kn" % P.WAIT_KN)
ax.set_xlim(0, xmax); ax.set_xlabel("speed over ground (knots)"); ax.set_ylabel("share of positions at or below"); ax.legend(fontsize=7, loc="lower right")
ax.set_title("ECDF per collection window (UTC)", fontsize=9, loc="left")
fig.tight_layout()
E.finish(fig, "fig01_sog_shape.png", "%d of %d positions sit below %.1f kn, and a long tail runs up to %.1f kn" % (n_zero, len(sog), P.WAIT_KN, sog.max()),
         "How is vessel speed distributed? (dashed line: waiting threshold)",
         "n = %d AIS positions, 3 collection windows. Flags: sog %s, row %s. Clock: UTC. Counts, bars start at zero." % (len(sog), P.SOG_FLAGS_MAIN, P.ROW_FLAGS_MAIN))
plt.show()
for bw in P.HIST_BINS_KN:
    h, _ = np.histogram(sog, bins=np.arange(0, xmax + bw, bw))
    print("speed bin %.1f kn: %d non-empty bins, first bin holds %.0f%% of positions" % (bw, int((h > 0).sum()), 100 * h[0] / h.sum()))

In [ ]:
WV = E.query(con, "SELECT point_key, observed_at, wave_height, wave_period FROM wave_reading_clean "
                  "WHERE wave_height IS NOT NULL AND wave_height_flag IN (?, ?, ?) ORDER BY point_key, observed_at", WAVE_F3)
wh = np.array([r["wave_height"] for r in WV], float)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
ax = axes[0]
ax.hist(wh, bins=np.arange(0, wh.max() + 0.04, 0.04), color="#4477aa", edgecolor="white")
ax.set_yscale("log"); ax.set_xlim(0, 1.6); ax.axvline(1.5, color="#cc3311", lw=1, ls="--")
ax.set_xlabel("significant wave height (m)"); ax.set_ylabel("hourly readings per bin (log scale)")
ax.set_title("bin width 0.04 m; dashed = 1.5 m threshold of the question", fontsize=8, loc="left")
ax = axes[1]
for pk in sorted({r["point_key"] for r in WV}):
    x, y = E.ecdf([r["wave_height"] for r in WV if r["point_key"] == pk]); ax.step(x, y, where="post", label="point %d" % pk)
ax.axvline(1.5, color="#cc3311", lw=1, ls="--"); ax.set_xlim(0, 1.6)
ax.set_xlabel("significant wave height (m)"); ax.set_ylabel("share of readings at or below"); ax.legend(fontsize=7, ncol=2, loc="lower right")
ax.set_title("ECDF per wave point (same axes)", fontsize=8, loc="left")
fig.tight_layout()
E.finish(fig, "fig02_wave_shape.png", ("Waves peak at %.2f m, %.1f times below the question's 1.5 m threshold" % (wh.max(), 1.5 / wh.max())) if wh.max() < 1.5 else ("Waves reach %.2f m, above the question's 1.5 m threshold in %d readings" % (wh.max(), int((wh > 1.5).sum()))),
         "How is wave height distributed at our 8 model points?",
         "n = %d hourly model readings, 8 points. Flags: wave_height %s. Clock: UTC. Counts, bars start at zero." % (len(wh), P.WAVE_FLAGS_MAIN))
plt.show()
for bw in P.HIST_BINS_M:
    h, _ = np.histogram(wh, bins=np.arange(0, wh.max() + bw, bw))
    print("wave bin %.2f m: %d non-empty bins, fullest bin holds %.0f%% of readings" % (bw, int((h > 0).sum()), 100 * h.max() / h.sum()))
print("readings above 1.5 m: %d; above 1.0 m: %d; maximum %.2f m" % ((wh > 1.5).sum(), (wh > 1.0).sum(), wh.max()))

## 2.4 Coverage, denominators and the flag filter
Coverage of the wave series per wave point and local day, of AIS per collection window, and of AIS positions near a wave point.
Coverage rules: a local-day wave mean needs at least 18 of 24 hours; a share over a group of positions needs at least 10 positions.

In [ ]:
print("Wave rows per wave point and UTC day (cleaned layer, all flags, NULL values counted separately):")
E.show(E.query(con, "SELECT point_key, substr(observed_at, 1, 10) AS utc_day, COUNT(*) AS rows_present, SUM(wave_height IS NOT NULL) AS values_present "
                    "FROM wave_reading_clean GROUP BY point_key, utc_day ORDER BY point_key, utc_day"))

In [ ]:
hours = defaultdict(set)          # (point, local date) -> local hours present
for r in WV:
    loc = E.to_local(r["observed_at"], TZ); r["ldate"], r["lhour"] = loc.date(), loc.hour
    hours[(r["point_key"], r["ldate"])].add(loc.hour)
POINTS = sorted({k[0] for k in hours}); LDAYS = sorted({k[1] for k in hours})
print("Local days (%s) with hours present per wave point, expected 24:" % P.TZ_NAME)
E.show([dict(local_day=str(d), weekday=d.strftime("%a"), **{"p%d" % pk: len(hours.get((pk, d), ())) for pk in POINTS}) for d in LDAYS])
KEEP = [d for d in LDAYS if all(len(hours.get((pk, d), ())) >= P.MIN_HOURS_PER_DAY for pk in POINTS)]
print("local days passing the rule (>= %d of 24 hours at every point): %s" % (P.MIN_HOURS_PER_DAY, [str(d) for d in KEEP]))
print("local days dropped:", [str(d) for d in LDAYS if d not in KEEP])
daily = {}                        # (point, local date) -> mean wave height
for (pk, d), hs in hours.items():
    vals = [r["wave_height"] for r in WV if r["point_key"] == pk and r["ldate"] == d]
    daily[(pk, d)] = (float(np.mean(vals)), len(vals))
print("\nMean of daily means per point, WITH and WITHOUT the 18-hour rule (flags: wave_height %s):" % (P.WAVE_FLAGS_MAIN,))
E.show([dict(point=pk, days_all=sum(1 for d in LDAYS if (pk, d) in daily), mean_all_days_m=round(float(np.mean([daily[(pk, d)][0] for d in LDAYS if (pk, d) in daily])), 3),
             days_rule=len(KEEP), mean_rule_m=round(float(np.mean([daily[(pk, d)][0] for d in KEEP])), 3)) for pk in POINTS])
print("\nAIS positions per vessel (how many vessels have k positions):", sorted(Counter(Counter(r["vessel_key"] for r in MP).values()).items()))
print("\nAIS positions with a wave point nearby, by radius (nearest point within the radius AND a wave reading in that hour):")
E.show([dict(radius_deg=rad, approx_km=round(rad * 111), positions=sum(1 for r in MP if r["dist_deg"] <= rad and r["wave_height"] is not None),
             vessels=len({r["vessel_key"] for r in MP if r["dist_deg"] <= rad and r["wave_height"] is not None}),
             of_total=len(MP)) for rad in P.RADIUS_SENSITIVITY])
print("\nAIS positions with NO wave reading in their hour (flag filter %s):" % (P.WAVE_FLAGS_MAIN,), sum(1 for r in MP if r["wave_height"] is None))

In [ ]:
print("TWO FLAG FILTERS on the key number (matched positions, radius %.1f deg):" % P.MATCH_RADIUS_DEG)
strict = E.query(con, clean_sql, (P.COS2, *E.pad(P.SOG_FLAGS_STRICT, 2), *E.pad(P.ROW_FLAGS_STRICT, 2), P.WAIT_KN, P.WAIT_KN, r2, *E.pad(P.WAVE_FLAGS_STRICT, 3)))[0]
E.show([dict(filter="main: sog %s, row %s, wave %s" % (P.SOG_FLAGS_MAIN, P.ROW_FLAGS_MAIN, P.WAVE_FLAGS_MAIN), **cl),
        dict(filter="strict: ok only everywhere", **strict)])
print("\nTwo wave filters on the mean wave height (all hourly readings):")
E.show([dict(filter="wave_height_flag = ok only", **E.query(con, "SELECT COUNT(*) AS n, ROUND(AVG(wave_height), 4) AS mean_m, ROUND(MAX(wave_height), 2) AS max_m FROM wave_reading_clean WHERE wave_height_flag = 'ok'")[0]),
        dict(filter="ok + unusual_high + stuck_value", **E.query(con, "SELECT COUNT(*) AS n, ROUND(AVG(wave_height), 4) AS mean_m, ROUND(MAX(wave_height), 2) AS max_m FROM wave_reading_clean WHERE wave_height_flag IN ('ok', 'unusual_high', 'stuck_value')")[0])])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [3, 2]})
days = [LDAYS[0] + timedelta(days=i) for i in range((LDAYS[-1] - LDAYS[0]).days + 1)]
grid = np.full((len(POINTS), len(days)), np.nan)
for i, pk in enumerate(POINTS):
    for j, d in enumerate(days):
        h = len(hours.get((pk, d), ()))
        if h: grid[i, j] = h
ax = axes[0]
cmap = plt.get_cmap("Blues").copy(); cmap.set_bad("#eeeeee")
im = ax.imshow(np.ma.masked_invalid(grid), aspect="auto", cmap=cmap, vmin=0, vmax=24)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        if not np.isnan(grid[i, j]): ax.text(j, i, int(grid[i, j]), ha="center", va="center", fontsize=6.5, color="black" if grid[i, j] < 16 else "white")
ax.set_xticks(range(len(days))); ax.set_xticklabels([d.strftime("%d %b") for d in days], rotation=60, fontsize=7)
ax.set_yticks(range(len(POINTS))); ax.set_yticklabels(["point %d" % p for p in POINTS], fontsize=7); ax.grid(False)
ax.set_title("hours with a wave reading per local day (grey = no data)", fontsize=8, loc="left")
ax = axes[1]
utc_days = [(datetime.strptime(LDAYS[0].isoformat(), "%Y-%m-%d") + timedelta(days=i)).strftime("%Y-%m-%d") for i in range((LDAYS[-1] - LDAYS[0]).days + 1)]
cnt = Counter(r["utc_date"] for r in MP)
ax.bar(range(len(utc_days)), [cnt.get(d, 0) for d in utc_days], color="#4477aa")
ax.set_xticks(range(len(utc_days))); ax.set_xticklabels([d[5:] for d in utc_days], rotation=60, fontsize=7)
ax.set_ylabel("AIS positions"); ax.set_title("AIS positions per UTC day", fontsize=8, loc="left")
fig.tight_layout()
E.finish(fig, "fig03_coverage.png", "Wave data covers %d of the %d local days between its first and last; AIS positions exist on %d days" % (len(LDAYS), len(days), len(cnt)),
         "What is each series made of, and where are the gaps?",
         "Wave: n = %d hourly readings, flags %s, clock %s. AIS: n = %d positions, flags sog %s / row %s, clock UTC. Bars start at zero." % (len(WV), P.WAVE_FLAGS_MAIN, P.TZ_NAME, len(MP), P.SOG_FLAGS_MAIN, P.ROW_FLAGS_MAIN))
plt.show()

## 2.5 Time
Clock: the wave series is aggregated in **local time (Europe/Istanbul, UTC+3)** because the question is about vessels, which follow
port and human schedules; AIS positions are described in both clocks. Edges: the first and last local days are partial.
Two scales: hour of day and day. Calendar: weekdays of the collection windows and Turkish public holidays.

In [ ]:
print("AIS collection windows in both clocks:")
E.show([dict(window_utc=w, n=by_window[w]["n"], local_date=str(next(r["local"].date() for r in MP if r["window"] == w)),
             weekday=next(r["local"].strftime("%A") for r in MP if r["window"] == w),
             local_hour=next(r["local"].strftime("%H:%M") for r in MP if r["window"] == w)) for w in WINDOWS])
print("\nHolidays in our local days:", {str(d): P.HOLIDAYS[str(d)] for d in LDAYS if str(d) in P.HOLIDAYS} or "none of %d local days is a listed holiday" % len(LDAYS))
print("Weekend days among the local days:", [d.strftime("%d %b %a") for d in LDAYS if d.weekday() >= 5] or "none")
print("Edge days (partial in local time):", [(str(d), min(len(hours.get((pk, d), ())) for pk in POINTS)) for d in LDAYS if d not in KEEP])
prof = {pk: [np.mean([r["wave_height"] for r in WV if r["point_key"] == pk and r["ldate"] in KEEP and r["lhour"] == h] or [np.nan]) for h in range(24)] for pk in POINTS}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
ax = axes[0]
for pk in POINTS: ax.plot(range(24), prof[pk], marker=".", lw=1, label="point %d" % pk)
ax.set_xlabel("local hour of day (%s)" % P.TZ_NAME); ax.set_ylabel("mean wave height (m)"); ax.set_xticks(range(0, 24, 3)); ax.set_ylim(bottom=0)
ax.set_title("hour of day (days passing the 18-hour rule)", fontsize=8, loc="left"); ax.legend(fontsize=6, ncol=2)
ax = axes[1]
for pk in POINTS:
    seg_x, seg_y = [], []
    for d in KEEP:
        if seg_x and (d - seg_x[-1]).days > 1:
            ax.plot(seg_x, seg_y, marker="o", ms=3, lw=1, color="C%d" % (pk % 10)); seg_x, seg_y = [], []
        seg_x.append(d); seg_y.append(daily[(pk, d)][0])
    ax.plot(seg_x, seg_y, marker="o", ms=3, lw=1, color="C%d" % (pk % 10))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b")); ax.tick_params(axis="x", rotation=45)
ax.set_xlabel("local day (%s)" % P.TZ_NAME); ax.set_title("day (gap = no data, lines are not drawn across it)", fontsize=8, loc="left")
fig.tight_layout()
hr_rng = max([np.nanmax(v) - np.nanmin(v) for v in prof.values() if not np.all(np.isnan(v))] or [0.0])
day_rng = max([max(daily[(pk, d)][0] for d in KEEP) - min(daily[(pk, d)][0] for d in KEEP) for pk in POINTS if KEEP] or [0.0])
E.finish(fig, "fig04_time_scales.png", ("Wave height moves more from day to day (up to %.2f m) than over the hour of day (up to %.2f m)" % (day_rng, hr_rng)) if day_rng > hr_rng else ("Wave height moves more over the hour of day (up to %.2f m) than from day to day (up to %.2f m)" % (hr_rng, day_rng)),
         "Which time scale carries the pattern in wave height?",
         "n = %d hourly readings on %d local days (>= %d of 24 hours at every point). Flags: wave_height %s. Clock: %s. y axis from zero." % (sum(1 for r in WV if r["ldate"] in KEEP), len(KEEP), P.MIN_HOURS_PER_DAY, P.WAVE_FLAGS_MAIN, P.TZ_NAME))
plt.show()

## 2.6 Relationships
Primary: wave height against vessel speed for AIS positions near a wave point (the project question). Plotted first, then Pearson and Spearman, then
drivers (leave one out, within each collection window). Secondary: wave height against wave period (n in the hundreds), the same checks.

In [ ]:
def corr_report(x, y, label):
    x, y = np.asarray(x, float), np.asarray(y, float)
    out = dict(set=label, n=len(x), pearson=round(E.pearson(x, y), 3), spearman=round(E.spearman(x, y), 3))
    out["note"] = "" if len(x) >= P.MIN_POSITIONS_FOR_CORR else "n below %d: printed, not interpreted" % P.MIN_POSITIONS_FOR_CORR
    return out
rows_c = []
for rad in P.RADIUS_SENSITIVITY:
    sub = [r for r in MP if r["dist_deg"] <= rad and r["wave_height"] is not None]
    rows_c.append(corr_report([r["wave_height"] for r in sub], [r["sog"] for r in sub], "radius <= %.1f deg" % rad))
print("wave height at the nearest point vs sog (flags: sog %s, row %s, wave %s):" % (P.SOG_FLAGS_MAIN, P.ROW_FLAGS_MAIN, P.WAVE_FLAGS_MAIN))
E.show(rows_c)
REL = [r for r in MP if r["dist_deg"] <= max(P.RADIUS_SENSITIVITY) and r["wave_height"] is not None]
PRIM = [r for r in REL if r["dist_deg"] <= P.MATCH_RADIUS_DEG]
if len(PRIM) >= 3:
    x, y = [r["wave_height"] for r in PRIM], [r["sog"] for r in PRIM]
    loo = [E.pearson(np.delete(x, i), np.delete(y, i)) for i in range(len(x))]
    print("\nLeave-one-out Pearson (primary radius): min %.3f, max %.3f" % (np.nanmin(loo), np.nanmax(loo)))
for w in WINDOWS:
    sub = [r for r in PRIM if r["window"] == w]
    print("within window %s: n=%d, Pearson=%.3f" % (w, len(sub), E.pearson([r["wave_height"] for r in sub], [r["sog"] for r in sub])) if len(sub) >= 3 else "within window %s: n=%d (too few)" % (w, len(sub)))
fig, ax = plt.subplots(figsize=(7, 4.2))
mk = {True: "o", False: "s"}
for i, w in enumerate(WINDOWS):
    for inner in (True, False):
        sub = [r for r in REL if r["window"] == w and (r["dist_deg"] <= P.MATCH_RADIUS_DEG) == inner]
        if sub:
            ax.scatter([r["wave_height"] for r in sub], [r["sog"] for r in sub], marker=mk[inner], s=34, color="C%d" % i,
                       facecolors="C%d" % i if inner else "none", label="%s, %s" % (w, "within %.1f deg" % P.MATCH_RADIUS_DEG if inner else "farther, up to %.1f deg" % max(P.RADIUS_SENSITIVITY)))
ax.axhline(P.WAIT_KN, color="#cc3311", lw=1, ls="--"); ax.set_xlim(0, 1.6); ax.set_ylim(bottom=-0.5)
ax.axvline(1.5, color="#888888", lw=1, ls=":")
ax.set_xlabel("significant wave height at the nearest point, UTC hour of the position (m)"); ax.set_ylabel("speed over ground (knots)")
ax.legend(fontsize=6.5, loc="upper right")
fig.tight_layout()
E.finish(fig, "fig05_wave_vs_sog.png", "Only %d positions lie within %.1f deg of a wave point, and the highest wave any of them meets is %.2f m" % (len(PRIM), P.MATCH_RADIUS_DEG, max([r["wave_height"] for r in PRIM] or [0])),
         "Do vessels near our wave points move slower when waves are higher?",
         "n = %d positions within %.1f deg (%d up to %.1f deg). Flags: sog %s, row %s, wave %s. Clock: UTC hour. Dashed: waiting threshold; dotted: 1.5 m threshold of the question." % (len(PRIM), P.MATCH_RADIUS_DEG, len(REL), max(P.RADIUS_SENSITIVITY), P.SOG_FLAGS_MAIN, P.ROW_FLAGS_MAIN, P.WAVE_FLAGS_MAIN))
plt.show()

In [ ]:
WP = E.query(con, "SELECT point_key, observed_at, wave_height, wave_period FROM wave_reading_clean "
                  "WHERE wave_height_flag IN (?, ?, ?) AND wave_period_flag = 'ok' AND wave_height IS NOT NULL AND wave_period IS NOT NULL ORDER BY point_key, observed_at", WAVE_F3)
h, p_ = np.array([r["wave_height"] for r in WP]), np.array([r["wave_period"] for r in WP])
print("wave height vs wave period (model readings; flags: wave_height %s, wave_period ok):" % (P.WAVE_FLAGS_MAIN,))
E.show([corr_report(h, p_, "all readings")])
top = h >= np.percentile(h, 97)
E.show([corr_report(h[~top], p_[~top], "without the highest 3%% of heights (%d readings)" % int(top.sum()))])
within = [E.pearson(h[[r["point_key"] == pk for r in WP]], p_[[r["point_key"] == pk for r in WP]]) for pk in sorted({r["point_key"] for r in WP})]
print("Pearson within each wave point: min %.3f, max %.3f (a pattern shared over time or place would change these)" % (np.nanmin(within), np.nanmax(within)))
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(h, p_, s=8, alpha=0.35, color="#4477aa")
edges = np.quantile(h, np.linspace(0, 1, 11)); edges = np.unique(edges)
mid, med = [], []
for a, b in zip(edges[:-1], edges[1:]):
    m = (h >= a) & (h <= b)
    if m.sum(): mid.append(float(np.median(h[m]))); med.append(float(np.median(p_[m])))
ax.plot(mid, med, color="#cc3311", marker="o", ms=4, lw=1.5, label="median per height decile")
ax.set_xlabel("significant wave height (m)"); ax.set_ylabel("wave period (s)"); ax.set_xlim(left=0); ax.set_ylim(bottom=0); ax.legend(fontsize=7)
fig.tight_layout()
E.finish(fig, "fig07_height_vs_period.png", "Wave height and wave period %s (Pearson %.2f, Spearman %.2f)" % ("rise together" if E.spearman(h, p_) > 0.3 else "show little relation", E.pearson(h, p_), E.spearman(h, p_)),
         "Does the wave model behave consistently? (a check of the wave series, not the project question)",
         "n = %d hourly readings, 8 points. Flags: wave_height %s, wave_period ok. Clock: UTC. Axes start at zero." % (len(h), P.WAVE_FLAGS_MAIN))
plt.show()

## 2.7 One comparison, with an interval, and a count of looks
Comparison: share of positions waiting (sog below 0.5 kn), first collection window against the later ones, over all AIS positions.
Unit: the vessel (one vessel in one group = one unit), bootstrap of vessels. This is about *when* vessels wait, not about waves.

In [ ]:
LOOKS = []
def look(label): LOOKS.append(label)
def unit_values(rows):
    per = defaultdict(list)
    for r in rows: per[r["vessel_key"]].append(1.0 if r["sog"] < P.WAIT_KN else 0.0)
    return np.array([np.mean(v) for v in per.values()])
for a, b in combinations(WINDOWS, 2):
    look("share waiting: %s vs %s" % (a, b)); look("median sog: %s vs %s" % (a, b))
if len(WINDOWS) >= 2:
    G1 = [r for r in MP if r["window"] == WINDOWS[0]]; G2 = [r for r in MP if r["window"] != WINDOWS[0]]
    look("share waiting: first window vs later windows (reported)")
    u1, u2 = unit_values(G1), unit_values(G2)
    d, lo, hi = E.bootstrap_diff(100 * u1, 100 * u2, P.N_BOOT, P.SEED, P.CI)
    c1, c2 = E.bootstrap_ci(100 * u1, P.N_BOOT, P.SEED, P.CI), E.bootstrap_ci(100 * u2, P.N_BOOT, P.SEED + 1, P.CI)
    print("Group A = %s (%d positions, %d vessels); group B = later windows (%d positions, %d vessels)" % (WINDOWS[0], len(G1), len(u1), len(G2), len(u2)))
    print("Share of vessels waiting: A %.1f%% [%.1f, %.1f], B %.1f%% [%.1f, %.1f]" % (100 * u1.mean(), *c1, 100 * u2.mean(), *c2))
    print("Difference A - B: %+.1f percentage points, %d%% bootstrap interval [%.1f, %.1f] (%d resamples of vessels, seed %d)" % (d, P.CI, lo, hi, P.N_BOOT, P.SEED))
    print("For comparison, the same interval computed on POSITIONS instead of vessels:", end=" ")
    pa = np.array([100.0 * (r["sog"] < P.WAIT_KN) for r in G1]); pb = np.array([100.0 * (r["sog"] < P.WAIT_KN) for r in G2])
    print("[%.1f, %.1f]" % E.bootstrap_diff(pa, pb, P.N_BOOT, P.SEED, P.CI)[1:])
    print("\nComposition check: latitude band (north = at or above %.1f N, where our wave points are)" % P.NORTH_LAT)
    comp = []
    for name, grp in (("A", G1), ("B", G2)):
        for band in ("north", "south"):
            sub = [r for r in grp if (r["latitude"] >= P.NORTH_LAT) == (band == "north")]
            comp.append(dict(group=name, band=band, positions=len(sub), vessels=len({r["vessel_key"] for r in sub}),
                             pct_waiting=round(100 * float(unit_values(sub).mean()), 1) if sub else None))
            look("share waiting: group %s, %s band" % (name, band))
    E.show(comp)
    signs = []
    for band in ("north", "south"):
        a_ = next(c for c in comp if c["group"] == "A" and c["band"] == band); b_ = next(c for c in comp if c["group"] == "B" and c["band"] == band)
        if a_["pct_waiting"] is not None and b_["pct_waiting"] is not None: signs.append(int(np.sign(a_["pct_waiting"] - b_["pct_waiting"])))
    print("overall difference sign: %+d; within-band signs: %s -> %s" % (int(np.sign(d)), signs, "reverses within bands (Simpson)" if signs and all(s == -np.sign(d) for s in signs) else "no reversal"))
    print("\nComparisons looked at while choosing this one: %d" % len(LOOKS)); print(*LOOKS, sep="\n")
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
    ax = axes[0]
    ax.bar([0, 1], [100 * u1.mean(), 100 * u2.mean()], color=["#4477aa", "#999999"], yerr=[[100 * u1.mean() - c1[0], 100 * u2.mean() - c2[0]], [c1[1] - 100 * u1.mean(), c2[1] - 100 * u2.mean()]], capsize=4)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["%s\n(n=%d vessels)" % (WINDOWS[0], len(u1)), "later windows\n(n=%d vessels)" % len(u2)]); ax.set_ylabel("vessels waiting (sog < %.1f kn), %%" % P.WAIT_KN); ax.set_ylim(0, 100)
    ax.set_title("overall, with %d%% bootstrap interval" % P.CI, fontsize=8, loc="left")
    ax = axes[1]
    for i, band in enumerate(("north", "south")):
        for j, name in enumerate(("A", "B")):
            c = next(c for c in comp if c["group"] == name and c["band"] == band)
            if c["pct_waiting"] is not None:
                ax.bar(i * 3 + j, c["pct_waiting"], color=["#4477aa", "#999999"][j]); ax.text(i * 3 + j, c["pct_waiting"] + 2, "n=%d" % c["vessels"], ha="center", fontsize=7)
    ax.set_xticks([0.5, 3.5]); ax.set_xticklabels(["north (>= %.0f N)" % P.NORTH_LAT, "south"]); ax.set_title("within latitude band (blue = %s, grey = later)" % WINDOWS[0], fontsize=8, loc="left")
    fig.tight_layout()
    E.finish(fig, "fig06_waiting_comparison.png", "Waiting share differs by %+.0f points between windows; the %d%% interval (%.0f to %+.0f) %s zero" % (d, P.CI, lo, hi, "includes" if lo <= 0 <= hi else "excludes"),
             "Do vessels wait more in the first collection window than in the later ones? (exploratory, %d looks)" % len(LOOKS),
             "Unit: vessel. n = %d positions (%d vessels). Flags: sog %s, row %s. Clock: UTC collection windows. Bars start at zero; error bars %d%% bootstrap, seed %d." % (len(MP), len({r['vessel_key'] for r in MP}), P.SOG_FLAGS_MAIN, P.ROW_FLAGS_MAIN, P.CI, P.SEED))
    plt.show()
else:
    print("only one collection window: no comparison possible")

## Decisions: parameters actually used, and proof that the database was not changed

In [ ]:
for k, v in PARAMS.items(): print("%-26s %s" % (k, v))
unchanged = table_digest(con) == RAW_BEFORE and clean_digest(con) == CLEAN_BEFORE
print("\nM2 and M3 tables identical before and after this notebook:", unchanged)
assert unchanged
con.close()